# Review the Schumann changes and prepare the publication project

Block 2, **40–90 minutes**, after the editing exercise. The mei-friend guide
covers 40–50; peer review 50–65; a short Git introduction 65–71; build/release
71–85; reflection 85–90. This notebook works on the existing local project.
It makes no GitHub changes, commits, pushes or releases automatically.

Keep `schumann-release-template.zip` beside it. It contains code/configuration,
notices and workflow files, not a source scan or inline binary payload.
The exported project is a **copy**; `edition/mei/work.mei` is preserved.

In [ ]:
%pip install -q "camat[validation,notebooks]==0.2.8" verovio==6.3.0 pypdf==6.10.0

In [1]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

"""Schumann review, transfer and release helpers. No GitHub mutations or model calls."""
import copy
import hashlib
import json
from pathlib import Path, PurePosixPath
import shutil
from urllib.parse import quote
import xml.etree.ElementTree as ET
from zipfile import ZipFile

REL_NS = "http://www.music-encoding.org/ns/mei"
REL_ID = "{http://www.w3.org/XML/1998/namespace}id"
REL_M = "{" + REL_NS + "}"


def release_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def release_write_once(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_bytes() != data:
            raise FileExistsError(f"Preserving a different existing file: {path}")
        return
    with path.open("xb") as output:
        output.write(data)


def release_json_once(path, value):
    release_write_once(path, (json.dumps(value, indent=2, ensure_ascii=False) + "\n").encode())


def make_review_record(project):
    project = Path(project)
    work = project / "edition/mei/work.mei"
    path = project / "edition/review.json"
    if path.exists():
        return path, json.loads(path.read_text())
    record = {"status": "not_reviewed", "reviewer": "", "date": "",
              "input_mei_sha256": release_hash(work), "intended_scope_mei_labels": [str(n) for n in range(1, 10)],
              "actually_reviewed_mei_labels": [], "source": "IMSLP 51556, printed p.8",
              "tasks": [{"task": name, "decision": "not_reviewed", "reason": ""}
                        for name in ["opening instruction/dynamic", "two opening slur phrases", "anchored editorial explanation"]],
              "validation_report": "edition/validation-review/run.json", "unresolved_issues": [],
              "whole_piece_approval": False, "model_contribution_and_human_corrections": ""}
    release_json_once(path, record)
    return path, record


def find_release_template():
    paths = [Path.cwd() / "schumann-release-template.zip",
             Path.cwd() / "sessions/02-building-an-edition/notebooks/schumann-release-template.zip"]
    return next((path for path in paths if path.is_file()), None)


def export_release_project(project, template, output):
    project, output = Path(project).resolve(), Path(output).resolve()
    if template is None:
        raise FileNotFoundError("Keep schumann-release-template.zip beside the notebook")
    mei = project / "edition/mei/work.mei"
    pages = project / "edition/pages/pages.json"
    review_path, review = make_review_record(project)
    identity = {"mei_sha256": release_hash(mei), "pages_sha256": release_hash(pages),
                "review_sha256": release_hash(review_path), "template_sha256": release_hash(template)}
    if output.exists():
        record = json.loads((output / "export.json").read_text())
        if record["identity"] != identity:
            raise FileExistsError("This export belongs to different inputs; use a new release attempt folder")
        print("Existing release project preserved; no files overwritten:", output)
        return record
    output.mkdir(parents=True)
    with ZipFile(template) as archive:
        manifest = json.loads(archive.read("template-manifest.json"))
        for name, expected in manifest["files_sha256"].items():
            parts = PurePosixPath(name).parts
            if not parts or name.startswith("/") or ".." in parts:
                raise ValueError("Unexpected template path")
            data = archive.read(name)
            if hashlib.sha256(data).hexdigest() != expected:
                raise ValueError(f"Template hash differs: {name}")
            release_write_once(output / name, data)
    release_write_once(output / "mei/work.mei", mei.read_bytes())
    exported_review = copy.deepcopy(review)
    validation = project / "edition/validation-review"
    if (validation / "run.json").is_file():
        for report_file in validation.rglob("*"):
            if report_file.is_file() and report_file.suffix in {".json", ".csv", ".txt"}:
                release_write_once(output / "validation-evidence" / report_file.relative_to(validation), report_file.read_bytes())
        exported_review["validation_report_in_working_project"] = exported_review.get("validation_report")
        exported_review["validation_report"] = "validation-evidence/run.json"
    release_json_once(output / "review.json", exported_review)
    page_manifest = json.loads(pages.read_text())
    exported_manifest = copy.deepcopy(page_manifest)
    exported_manifest["pdf"] = "../local/schumann-op68-no10-source-page.pdf"
    release_json_once(output / "pages/pages.json", exported_manifest)
    for page in page_manifest["pages"]:
        source = (pages.parent / page["image"]).resolve()
        if release_hash(source) != page["sha256"]:
            raise ValueError("Page image differs from the linked source manifest")
        release_write_once(output / "pages" / Path(page["image"]).name, source.read_bytes())
    pdf = (pages.parent / page_manifest["pdf"]).resolve()
    if pdf.is_file():
        if release_hash(pdf) != page_manifest["pdf_sha256"]:
            raise ValueError("Source excerpt differs from page manifest")
        release_write_once(output / "local/schumann-op68-no10-source-page.pdf", pdf.read_bytes())
    record = {"prepared_on": "2026-10-26", "identity": identity, "source_mei": str(mei),
              "working_mei_unchanged": release_hash(mei) == identity["mei_sha256"],
              "review_status": review["status"], "review_hash_matches_input": review["input_mei_sha256"] == identity["mei_sha256"],
              "human_approval_created": False, "github_mutations": 0,
              "media": "Local PDF/PNG copied for rehearsal; .gitignore excludes them and the public build omits them"}
    release_json_once(output / "export.json", record)
    return record


def make_friend_transfer(project, base_url="http://127.0.0.1:8781/"):
    project = Path(project).resolve()
    source = project / "edition/mei/work.mei"
    root = ET.parse(source).getroot()
    targets = []
    for graphic in root.findall("m:music/m:facsimile/m:surface/m:graphic", {"m": REL_NS}):
        old = graphic.get("target")
        image = (source.parent / old).resolve()
        if image.parent != project / "edition/pages":
            raise ValueError("Expected image in this edition's pages folder")
        url = base_url.rstrip("/") + "/pages/" + quote(image.name)
        targets.append({"original": old, "transfer": url, "image_sha256": release_hash(image)})
        graphic.set("target", url)
    if not targets:
        raise ValueError("No facsimile graphic found")
    output = source.with_name("mei-friend-transfer.mei")
    release_write_once(output, ET.tostring(root, encoding="UTF-8", xml_declaration=True))
    report = {"source_mei_sha256": release_hash(source), "transfer_sha256": release_hash(output),
              "targets": targets, "canonical_working_file": str(source),
              "warning": "Transfer-only localhost paths; restore them before publication"}
    release_json_once(project / "edition/mei-friend-transfer.json", report)
    return output, report


def check_friend_roundtrip(original, saved, output, transfer_record=None):
    """Save a separate restored candidate; report identity/music/link preservation."""
    original, saved, output = Path(original), Path(saved), Path(output)
    before, after = ET.parse(original).getroot(), ET.parse(saved).getroot()
    if transfer_record:
        if release_hash(original) != transfer_record["source_mei_sha256"]:
            raise ValueError("Original changed since the transfer was made")
        graphics = after.findall("m:music/m:facsimile/m:surface/m:graphic", {"m": REL_NS})
        if len(graphics) != len(transfer_record["targets"]):
            raise ValueError("Graphic count changed")
        for graphic, target in zip(graphics, transfer_record["targets"]):
            if graphic.get("target") != target["transfer"]:
                raise ValueError("Unexpected graphic target; inspect before restoring")
            graphic.set("target", target["original"])
    old_ids = {e.get(REL_ID) for e in before.iter() if e.get(REL_ID)}
    new_ids = [e.get(REL_ID) for e in after.iter() if e.get(REL_ID)]
    def signature(root):
        body = root.find("m:music/m:body", {"m": REL_NS})
        if body is None:
            raise ValueError("No musical body found")
        return [(e.tag, sorted(e.attrib.items()), (e.text or "").strip() if e.tag != REL_M + "annot" else "")
                for e in body.iter()]
    def facsimile(root):
        return [(e.tag, sorted(e.attrib.items())) for e in root.iter()
                if e.tag in {REL_M + tag for tag in ["facsimile", "surface", "graphic", "zone"]}]
    report = {"existing_ids_preserved": old_ids.issubset(new_ids), "ids_unique": len(new_ids) == len(set(new_ids)),
              "musical_structure_and_controls_preserved": signature(before) == signature(after),
              "facsimile_preserved_after_restore": facsimile(before) == facsimile(after),
              "original_sha256": release_hash(original), "saved_sha256": release_hash(saved),
              "human_source_review": "not_assessed", "metadata_and_annotation_prose": "Inspect the diff separately"}
    if not all(report[name] for name in ["existing_ids_preserved", "ids_unique", "musical_structure_and_controls_preserved", "facsimile_preserved_after_restore"]):
        raise ValueError("Round-trip structure changed: " + json.dumps(report))
    release_write_once(output, ET.tostring(after, encoding="UTF-8", xml_declaration=True))
    return report

In [2]:
import subprocess
import sys
from IPython.display import display

PROJECT = (Path.cwd() / "schumann_edition_work").resolve()
WORK_MEI = PROJECT / "edition/mei/work.mei"
if not WORK_MEI.is_file():
    raise FileNotFoundError(f"Run the linking/editing notebooks, or set PROJECT: {WORK_MEI}")
WORK_HASH = release_hash(WORK_MEI)
print("Working file:", WORK_MEI)
print("SHA-256:", WORK_HASH)

Working file: /tmp/camat-schumann-release-verified/schumann_edition_work/edition/mei/work.mei
SHA-256: 0f9ecfec0a8d133932b3c75449227c5951749baeb41c8e043e377c3aeabca585


## 40–50 — mei-friend alternative (optional transfer copy)

Follow `mei-friend.md`: open a copy, select the pickup, inspect an annotation,
validate and Save MEI as a new file. Keep the notebook source panel available.
The hosted editor cannot automatically use this project's local image paths.

For a rehearsed CORS/loopback route, enable the following transfer-copy cell.
It substitutes absolute page URLs in a separate file, recording originals.
Serve only the edition's transfer MEI and page resources with the supplied
`tools/serve_mei_friend_preview.py` script from the lecture checkout. Browser
restrictions may still require the separate-source fallback. Stop the server
after the comparison. Never publish the localhost transfer copy.

In [3]:
MAKE_FRIEND_TRANSFER = False
if MAKE_FRIEND_TRANSFER:
    TRANSFER_MEI, TRANSFER = make_friend_transfer(PROJECT)
    print("Transfer copy:", TRANSFER_MEI)
    print("Local URL: http://127.0.0.1:8781/mei/mei-friend-transfer.mei")
else:
    print("Use File → Open file on a copy in mei-friend; the canonical working file is unchanged.")

Use File → Open file on a copy in mei-friend; the canonical working file is unchanged.


After saving from mei-friend, this optional cell restores recorded graphic
targets into another separate candidate and checks IDs/music/facsimile. Use
the transfer record only if that was the imported file. Inspect metadata and
annotation prose in a diff, then validate. Nothing promotes it to `work.mei`.

In [4]:
FRIEND_SAVED = None  # e.g. Path('/absolute/path/to/downloaded-save.mei')
if FRIEND_SAVED is not None:
    restored = WORK_MEI.with_name("mei-friend-restored.mei")
    transfer_record = json.loads((PROJECT / "edition/mei-friend-transfer.json").read_text()) if MAKE_FRIEND_TRANSFER else None
    print(check_friend_roundtrip(WORK_MEI, FRIEND_SAVED, restored, transfer_record))
    print("Inspect and validate this separate candidate:", restored)

## 50–65 — Peer review and validation

Follow `peer-review.md`. Check source reading, encoded attachment and display
separately. Record the exact passage and decisions that a person actually
reviewed. The default record stays `not_reviewed`; running cells supplies no
source approval. Fill in the existing record in the editor, then rerun its
display cell. After a musical edit, update the hash and perform the review
again. The optional CAMAT parse consumer is not requested by this notebook.

In [5]:
REVIEW_PATH, REVIEW = make_review_record(PROJECT)
print("Fill in:", REVIEW_PATH)
print(json.dumps(REVIEW, indent=2))
print("Review hash matches current MEI:", REVIEW["input_mei_sha256"] == release_hash(WORK_MEI))

Fill in: /tmp/camat-schumann-release-verified/schumann_edition_work/edition/review.json
{
  "status": "not_reviewed",
  "reviewer": "",
  "date": "",
  "input_mei_sha256": "0f9ecfec0a8d133932b3c75449227c5951749baeb41c8e043e377c3aeabca585",
  "intended_scope_mei_labels": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9"
  ],
  "actually_reviewed_mei_labels": [],
  "source": "IMSLP 51556, printed p.8",
  "tasks": [
    {
      "task": "opening instruction/dynamic",
      "decision": "not_reviewed",
      "reason": ""
    },
    {
      "task": "two opening slur phrases",
      "decision": "not_reviewed",
      "reason": ""
    },
    {
      "task": "anchored editorial explanation",
      "decision": "not_reviewed",
      "reason": ""
    }
  ],
  "validation_report": "edition/validation-review/run.json",
  "unresolved_issues": [],
  "whole_piece_approval": false,
  "model_contribution_and_human_corrections": ""
}
Review hash matches current MEI: True


In [6]:
VALIDATION_DIR = PROJECT / "edition/validation-review"
validation = subprocess.run([sys.executable, "-m", "camat.mei_validation", str(WORK_MEI),
                             "--output-dir", str(VALIDATION_DIR), "--verovio"],
                            text=True, capture_output=True, timeout=180)
print(validation.stdout)
if validation.returncode:
    print(validation.stderr)
    raise RuntimeError("Required validation did not pass; inspect the reports")
run = json.loads((VALIDATION_DIR / "run.json").read_text())
print("Conformance:", run["conformance"])
print("Consumer diagnostics:", run["consumer_diagnostics"])

MEI conformance: passed; consumers: incomplete

Conformance: passed
Consumer diagnostics: incomplete


## 65–71 — Git/GitHub in one small cycle

Use `git-introduction.md`: local repository → inspect diff → stage selected
files → commit → push to the prepared remote. Save, commit, push, tag/release
and Pages deployment are different steps. Account setup happens before class.
This notebook deliberately runs no Git mutations; a local-only commit and an
instructor publication demonstration are the account-free fallback.

## 71–85 — Export, build and inspect the viewer

Choose a new release attempt folder below. The export copies the current MEI,
review and page manifest, and includes the local image/PDF in ignored folders
for rehearsal. Public builds omit those pixels and let visitors open the exact
one-page PDF locally. The source instructions and retrieval script preserve
the original/excerpt mapping and hashes. Check the title, edition statement,
credits, scope and unique pair storage prefix before committing the project.

In [7]:
RELEASE_PROJECT = PROJECT / "publication-01"
EXPORT = export_release_project(PROJECT, find_release_template(), RELEASE_PROJECT)
print(json.dumps(EXPORT, indent=2))
print("Review status remains:", EXPORT["review_status"])
assert release_hash(WORK_MEI) == WORK_HASH, "The original working file changed during export"

{
  "prepared_on": "2026-10-26",
  "identity": {
    "mei_sha256": "0f9ecfec0a8d133932b3c75449227c5951749baeb41c8e043e377c3aeabca585",
    "pages_sha256": "6c2e187e97cc85d2a782130d05634a56ab88029068c49bd790a05108f8d2331c",
    "review_sha256": "48948f4a837640b6c823f53fd3ac5a675a6e97c60f0b788915fa7b8f72128fe6",
    "template_sha256": "b23dec8fdc5b5f41b1124c218e906ca0b31ad6178c41bbd1dac1066883f68eb6"
  },
  "source_mei": "/tmp/camat-schumann-release-verified/schumann_edition_work/edition/mei/work.mei",
  "working_mei_unchanged": true,
  "review_status": "not_reviewed",
  "review_hash_matches_input": true,
  "human_approval_created": false,
  "github_mutations": 0,
  "media": "Local PDF/PNG copied for rehearsal; .gitignore excludes them and the public build omits them"
}
Review status remains: not_reviewed


In [8]:
check = subprocess.run([sys.executable, str(RELEASE_PROJECT / "scripts/check_edition.py")],
                       text=True, capture_output=True, timeout=180)
print(check.stdout)
if check.returncode:
    print(check.stderr)
    raise RuntimeError("Exported edition validation failed")
build = subprocess.run([sys.executable, str(RELEASE_PROJECT / "scripts/build_site.py")],
                       text=True, capture_output=True, timeout=120)
print(build.stdout)
if build.returncode:
    print(build.stderr)
    raise RuntimeError("Publication build failed")
published_mei = RELEASE_PROJECT / "_site/mei/work.mei"
assert release_hash(published_mei) == release_hash(WORK_MEI)
print("Downloadable MEI matches the working MEI byte-for-byte.")
print("Public build contains no PDF or PNG:", not any(p.suffix.lower() in {'.pdf', '.png'} for p in (RELEASE_PROJECT / '_site').rglob('*')))

{
  "scope": "Local: required conformance checks and Verovio, including local images",
  "checked": [
    "xml_references",
    "relaxng",
    "schematron",
    "input_hashes",
    "verovio",
    "resources"
  ],
  "failed": [],
  "conformance": "passed",
  "xml_reference_status": "passed",
  "expected_missing_local_graphics": [],
  "consumer_diagnostics": "incomplete",
  "source_fidelity": "Requires a recorded human source comparison"
}

Built /tmp/camat-schumann-release-verified/schumann_edition_work/publication-01/_site; MEI SHA-256 0f9ecfec0a8d133932b3c75449227c5951749baeb41c8e043e377c3aeabca585; source commit not yet committed

Downloadable MEI matches the working MEI byte-for-byte.
Public build contains no PDF or PNG: True


Preview from a terminal (replace the path with the printed release project):

```sh
python -m http.server --bind 127.0.0.1 --directory /absolute/path/to/publication-01/_site 8000
```

Open `http://127.0.0.1:8000/#landmann/1`, then `/6` and `/21`. Check the
annotation and downloaded MEI, and open the exact one-page PDF. Serve over
HTTP; the viewer uses JavaScript modules and external Verovio/pdf.js libraries.
On Jupyter4NFDI, use its documented port proxy instead of a local VS Code URL.

The project README and `RELEASE.md` cover GitHub Pages setup, deployment and
tagging the checked commit. `publication.json` records the source commit,
dirty state, MEI hash and review. GitHub Actions checks text/schema/references/
rendering without local images, explicitly accepting only the declared missing
graphic files. Raw full conformance can be failed/incomplete; full local image
validation remains a separate check, and other reference errors still fail CI.

## 85–90 — Reflect and submit

Show one consequential edit and its evidence, one unresolved limitation,
and what this encoding now makes analysable. Record repository, commit/tag,
viewer and tested bar-link URLs only after those actions actually happened.
If remote setup is pending, submit the local project and commit instead.
Keep the scan separate, and preserve the declared scope and notices.